# 02 — Multi-band synthetic cube + `write_cube_fits`

Builds three `FieldObservation`s with **tabulated throughput**, fits one shared `hashgrid_spectral_field`, renders a dense cube via `render_field_cube`, and writes a FITS cube with `write_cube_fits` (round-trip read with `SpectralReader`).

Also shows that an `ExposurePose` with non-zero `delta_ra_deg` changes the rendered patch (bundle-adjustment hook).

**Prerequisite:** install an explicitly selected torchsky build into the harness `test` environment. Notebook execution uses the installed subject and does not add a source checkout to `sys.path`. From the harness root, install the optional notebook UI once with `pixi run -e test -- python -m pip install jupyterlab matplotlib`, then launch with `pixi run -e test -- jupyter lab notebooks/02_multiband_hyperspectral_synthetic.ipynb`. See the [torchsky spectral guide](https://github.com/astroai/torchsky/blob/main/docs/spectral.md) and [field guide](https://github.com/astroai/torchsky/blob/main/docs/field.md).

In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

import torch
import matplotlib.pyplot as plt

from torchsky.field import (
    estimate_domain,
    fit as fit_field_model,
    model as make_model,
    render,
)
from torchsky.scene import FieldObservation
from torchsky.map.exposure_pose import ExposurePose  # not yet public: docs/subject-api-gaps.md
from torchsky.spectral import SpectralAxis, SpectralReader, write_cube_fits


def band_observation(
    *,
    image: torch.Tensor,
    header: dict,
    wave0: float,
    wave1: float,
    center: float,
    band_name: str,
) -> FieldObservation:
    wl = torch.linspace(wave0, wave1, 64, dtype=torch.float64)
    half = 0.5 * (wave1 - wave0)
    tp = torch.clamp(1.0 - (wl - center).abs() / max(half, 1.0), min=0.0)
    return FieldObservation(
        image=image,
        header=header,
        spectral_min=float(wave0),
        spectral_max=float(wave1),
        transmission_wavelength=wl.to(dtype=torch.float32),
        transmission_throughput=tp.to(dtype=torch.float32),
        band_name=band_name,
    )


header = {
    "NAXIS": 2,
    "NAXIS1": 24,
    "NAXIS2": 24,
    "CTYPE1": "RA---TAN",
    "CTYPE2": "DEC--TAN",
    "CRPIX1": 12.5,
    "CRPIX2": 12.5,
    "CRVAL1": 150.0,
    "CRVAL2": 2.0,
    "CDELT1": -0.2 / 3600.0,
    "CDELT2": 0.2 / 3600.0,
    "CUNIT1": "deg",
    "CUNIT2": "deg",
}

torch.manual_seed(7)
template = FieldObservation(
    image=torch.zeros(24, 24, dtype=torch.float32),
    header=header,
    spectral_min=5000.0,
    spectral_max=5000.0,
    transmission_center=5000.0,
)
domain = estimate_domain([template])
truth = make_model(
    "hashgrid_spectral_field",
    domain,
    hidden_dim=48,
    num_levels=4,
    log2_hashmap_size=14,
)

observations = []
centers = (4800.0, 5200.0, 5600.0)
width = 120.0
for name, c in zip(("g", "r", "i"), centers, strict=True):
    w0, w1 = c - width * 0.5, c + width * 0.5
    blank = FieldObservation(
        image=torch.zeros(24, 24, dtype=torch.float32),
        header=header,
        spectral_min=w0,
        spectral_max=w1,
        transmission_center=c,
    )
    with torch.no_grad():
        img = render(truth, blank, num_spectral_samples=12).detach()
    observations.append(
        band_observation(
            image=img,
            header=header,
            wave0=w0,
            wave1=w1,
            center=c,
            band_name=name,
        )
    )

torch.manual_seed(11)
model = make_model(
    "hashgrid_spectral_field",
    estimate_domain(observations),
    hidden_dim=48,
    num_levels=4,
    log2_hashmap_size=14,
)
fit = fit_field_model(observations, model, steps=100, lr=2e-3, num_spectral_samples=12)
print("loss", fit.initial_loss, "->", fit.final_loss)

cube_axis = SpectralAxis(
    values=torch.linspace(4750.0, 5650.0, 20, dtype=torch.float64),
    unit="Angstrom",
    type="WAVE",
)
with torch.no_grad():
    cube = render(model, target="cube", target_header=header, spectral_axis=cube_axis)

tmpdir = Path(tempfile.mkdtemp(prefix="torchsky_nb02_"))
cube_path = tmpdir / "synthetic_hyperspectral.fits"
spatial_hdr = {
    k: v for k, v in header.items() if not str(k).upper().startswith("NAXIS")
}
spatial_hdr["NAXIS"] = 2
spatial_hdr["NAXIS1"] = header["NAXIS1"]
spatial_hdr["NAXIS2"] = header["NAXIS2"]
write_cube_fits(
    str(cube_path),
    cube.cpu(),
    spatial_header=spatial_hdr,
    spectral_axis=cube_axis,
    overwrite=True,
)
roundtrip = SpectralReader.read_data_cube(str(cube_path), hdu=0)
torch.testing.assert_close(roundtrip.data, cube.cpu(), rtol=1e-4, atol=1e-4)
print("wrote", cube_path)

pose = ExposurePose()
with torch.no_grad():
    pose.delta_ra_deg.fill_(0.5 / 3600.0)
with torch.no_grad():
    base = render(model, observations[0], num_spectral_samples=12)
    shifted = render(
        model, observations[0], num_spectral_samples=12, exposure_pose=pose
    )

fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].imshow(base.numpy(), origin="lower")
ax[0].set_title("render (identity pose)")
ax[1].imshow(shifted.numpy(), origin="lower")
ax[1].set_title("render (ExposurePose)")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()